# Adverse Drug Reaction (ADR) Data Analysis

This notebook performs descriptive pharmacovigilance analysis on the supplied CSV:
1. Load and validate the ADR dataset.
2. Calculate reporting frequency by suspected drug and reaction type.
3. Screen drug–reaction pairs by serious-report ratio.
4. Visualize ADR report counts across patient age brackets.
5. Produce an evidence-limited safety signal summary.

**Important:** These are descriptive report frequencies, not incidence rates or proof of causality. Conventional disproportionality methods require suitable comparator data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Update this path if running outside the provided environment
DATA_PATH = "adverse_drug_reactions_v2(1).csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().rename("Missing values"))

In [ ]:
# Parse dates and define age brackets
df["ReceivedDate"] = pd.to_datetime(df["ReceivedDate"], errors="coerce")
age_bins = [0, 17, 29, 44, 59, 74, 120]
age_labels = ["0–17", "18–29", "30–44", "45–59", "60–74", "75+"]
df["AgeBracket"] = pd.cut(df["PatientAge"], bins=age_bins, labels=age_labels)
n = len(df)
print(f"Total reports: {n}")
print("Date range:", df["ReceivedDate"].min(), "to", df["ReceivedDate"].max())

In [ ]:
# Reporting frequency by suspected drug and adverse reaction
drug_frequency = (df.groupby("SuspectedDrug").size().rename("Reports").reset_index())
drug_frequency["Reporting frequency (%)"] = drug_frequency["Reports"] / n * 100
drug_frequency = drug_frequency.sort_values("Reports", ascending=False)

reaction_frequency = (df.groupby("AdverseReaction").size().rename("Reports").reset_index())
reaction_frequency["Reporting frequency (%)"] = reaction_frequency["Reports"] / n * 100
reaction_frequency = reaction_frequency.sort_values("Reports", ascending=False)

display(drug_frequency)
display(reaction_frequency)

In [ ]:
# Drug x reaction frequency index (% of all submitted reports)
drug_reaction_counts = pd.crosstab(df["SuspectedDrug"], df["AdverseReaction"])
drug_reaction_index = drug_reaction_counts / n * 100
display(drug_reaction_index.round(2))

plt.figure(figsize=(10, 5))
plt.imshow(drug_reaction_index.values, aspect="auto")
plt.colorbar(label="Share of all reports (%)")
plt.xticks(range(len(drug_reaction_index.columns)), drug_reaction_index.columns, rotation=35, ha="right")
plt.yticks(range(len(drug_reaction_index.index)), drug_reaction_index.index)
plt.title("Drug–reaction reporting frequency index")
plt.tight_layout()
plt.show()

In [ ]:
# Seriousness ratio per drug–reaction pair
pair_summary = (df.groupby(["SuspectedDrug", "AdverseReaction"])
    .agg(Reports=("ReportID", "count"),
         SeriousReports=("Serious", lambda s: (s == "Yes").sum()))
    .reset_index())
pair_summary["Severity ratio (%)"] = pair_summary["SeriousReports"] / pair_summary["Reports"] * 100
pair_summary["Reporting frequency (%)"] = pair_summary["Reports"] / n * 100

# Exploratory threshold: at least 3 reports per pair
high_risk = (pair_summary[pair_summary["Reports"] >= 3]
             .sort_values(["Severity ratio (%)", "Reports"], ascending=[False, False]))
display(high_risk)
print("Note: severity ratio is descriptive, not a validated causal-risk measure.")

In [ ]:
# Visualize report frequency across age brackets
age_frequency = (df.groupby("AgeBracket", observed=False).size()
                 .rename("Reports").reset_index())
age_frequency["Frequency (%)"] = age_frequency["Reports"] / n * 100
display(age_frequency)

plt.figure(figsize=(8, 4.5))
plt.bar(age_frequency["AgeBracket"].astype(str), age_frequency["Reports"])
plt.xlabel("Patient age bracket")
plt.ylabel("Number of reports")
plt.title("ADR report frequency across patient age brackets")
plt.tight_layout()
plt.show()

In [ ]:
# Compact safety signal detection summary
serious_n = (df["Serious"] == "Yes").sum()
unexpected_n = (df["Expectedness"] == "Unexpected").sum()
print(f"Total reports: {n}")
print(f"Serious reports: {serious_n} ({serious_n/n*100:.1f}%)")
print(f"Unexpected reports: {unexpected_n} ({unexpected_n/n*100:.1f}%)")
print("\nHighest severity-ratio pairs (minimum 3 reports):")
display(high_risk[["SuspectedDrug", "AdverseReaction", "Reports", "SeriousReports", "Severity ratio (%)"]].head(10))

print("\nInterpretation:")
print("- Prioritize serious and unexpected cases for clinical case review.")
print("- Check narratives, onset, dechallenge/rechallenge, dose, indication and concomitant medicines.")
print("- The supplied data contain only one reaction type per suspected drug; broader comparator data are needed.")
print("- Do not infer incidence, causality, or comparative drug risk from these report counts.")

In [ ]:
# Export analysis tables
drug_frequency.to_csv("adr_drug_frequency.csv", index=False)
reaction_frequency.to_csv("adr_reaction_frequency.csv", index=False)
pair_summary.to_csv("adr_drug_reaction_severity.csv", index=False)
age_frequency.to_csv("adr_age_bracket_frequency.csv", index=False)
drug_reaction_index.to_csv("adr_drug_reaction_frequency_index_pct.csv")
print("CSV outputs exported.")